# Système de Recommandation — TF-IDF + Filtrage par Contenu

## Approche
Le **filtrage par contenu** recommande des produits **similaires** à ceux qu'un utilisateur a déjà aimés,
en se basant sur le **texte des avis** plutôt que sur les comportements d'autres utilisateurs.

**Avantage principal** : résout partiellement le problème du **cold start** —
même un nouvel utilisateur avec peu d'historique peut recevoir des recommandations.

## Pipeline
1. Charger les données et agréger les avis par produit
2. Vectoriser les textes avec **TF-IDF**
3. Calculer la **similarité cosinus** entre produits
4. Recommander les produits les plus similaires
5. Évaluer le modèle

## Étape 1 — Chargement et agrégation des données

Pour le filtrage par contenu, on travaille au niveau **produit** (et non utilisateur).
On concatène tous les avis d'un même produit en un seul texte.

In [1]:
import pandas as pd
import numpy as np
import json

# On utilise reviews_filtered : texte + notes, produits avec min 5 avis
df = pd.read_csv("../data/processed/reviews_filtered.csv")

print(f"Données chargées : {len(df):,} avis")
print(f"Produits uniques : {df['asin'].nunique():,}")
print(f"Colonnes : {list(df.columns)}")

Données chargées : 5,609 avis
Produits uniques : 730
Colonnes : ['user_id', 'asin', 'rating', 'text', 'verified_purchase', 'date', 'year']


In [2]:
# Agrégation : 1 texte par produit (concaténation de tous ses avis)
produits = df.groupby('asin')['text'].apply(lambda x: ' '.join(x.dropna())).reset_index()
produits.columns = ['asin', 'texte_agrege']

print(f"Nombre de produits : {len(produits):,}")
print(f"\nExemple — produit {produits['asin'].iloc[0]} :")
print(produits['texte_agrege'].iloc[0][:300], "...")

Nombre de produits : 730

Exemple — produit B005AL5H9S :
I'm a big fan of the style of this brand and have been for years. There's something about the vintage aesthetic that they capture so perfectly that always draws me in. I wasn't sure what I would think about their actual lipsticks however because of the bullet style is so unusual, but I really love t ...


## Étape 2 — Vectorisation TF-IDF

On transforme chaque texte agrégé en **vecteur numérique**.

Paramètres choisis :
- `max_features=5000` : on garde les 5 000 mots les plus discriminants
- `min_df=2` : un mot doit apparaître dans au moins 2 produits (évite les fautes de frappe)
- `max_df=0.85` : on ignore les mots présents dans plus de 85% des produits (trop banals)
- `stop_words='english'` : on retire les mots vides (the, a, is...)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorisation TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,  # top 5000 mots
    min_df=2,           # mot présent dans au moins 2 produits
    max_df=0.85,        # mot présent dans max 85% des produits
    stop_words='english'
)

matrice_tfidf = tfidf.fit_transform(produits['texte_agrege'])

print(f"Forme de la matrice TF-IDF : {matrice_tfidf.shape}")
print(f"→ {matrice_tfidf.shape[0]} produits × {matrice_tfidf.shape[1]} mots")
print(f"\nExemple — top 10 mots du produit {produits['asin'].iloc[0]} :")
mots = tfidf.get_feature_names_out()
vecteur = matrice_tfidf[0].toarray()[0]
top10 = sorted(zip(mots, vecteur), key=lambda x: x[1], reverse=True)[:10]
for mot, score in top10:
    print(f"  {mot:20s} → {score:.4f}")

Forme de la matrice TF-IDF : (730, 5000)
→ 730 produits × 5000 mots

Exemple — top 10 mots du produit B005AL5H9S :
  vintage              → 0.4267
  red                  → 0.3910
  lipstick             → 0.3909
  lipsticks            → 0.2426
  lips                 → 0.1430
  color                → 0.1254
  blue                 → 0.1209
  stain                → 0.1198
  tube                 → 0.1052
  shape                → 0.1034


## Étape 3 — Similarité cosinus entre produits

La **similarité cosinus** mesure l'angle entre deux vecteurs TF-IDF.
- Score = **1.0** → produits identiques (même vocabulaire)
- Score = **0.0** → produits sans aucun mot en commun

On calcule la similarité entre **tous les produits** pour pouvoir ensuite trouver les plus proches.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Matrice de similarité : shape (730, 730)
# sim[i][j] = similarité entre produit i et produit j
sim = cosine_similarity(matrice_tfidf)

print(f"Forme de la matrice de similarité : {sim.shape}")
print(f"\nExemple — similarité du produit 0 avec les 5 premiers produits :")
for i in range(5):
    print(f"  {produits['asin'].iloc[0]} ↔ {produits['asin'].iloc[i]} : {sim[0][i]:.4f}")

Forme de la matrice de similarité : (730, 730)

Exemple — similarité du produit 0 avec les 5 premiers produits :
  B005AL5H9S ↔ B005AL5H9S : 1.0000
  B005AL5H9S ↔ B009GPP546 : 0.0645
  B005AL5H9S ↔ B00BETY3EU : 0.1162
  B005AL5H9S ↔ B00O2FGBJS : 0.0723
  B005AL5H9S ↔ B00TK0VV68 : 0.0549


## Étape 4 — Fonction de recommandation par contenu

La fonction prend un **ASIN produit** en entrée et retourne les N produits les plus similaires.

**Logique :**
1. Trouver l'index du produit dans notre liste
2. Récupérer sa ligne dans la matrice de similarité
3. Trier par score décroissant
4. Retourner le Top N (en excluant le produit lui-même)

In [5]:
def recommander_contenu(asin, n_reco=5):
    """
    Recommande N produits similaires à un produit donné (par son ASIN).
    Basé sur la similarité du contenu textuel des avis.
    """
    # Vérifier que le produit existe
    if asin not in produits['asin'].values:
        print(f"Produit {asin} non trouvé.")
        return []

    # Index du produit dans notre liste
    idx = produits[produits['asin'] == asin].index[0]

    # Scores de similarité avec tous les autres produits
    scores = list(enumerate(sim[idx]))

    # Trier par score décroissant, exclure le produit lui-même (score=1.0)
    scores_tries = sorted(scores, key=lambda x: x[1], reverse=True)[1:n_reco+1]

    print(f"Produits similaires à {asin} :\n")
    print(f"{'Rang':<5} {'ASIN':<15} {'Similarité'}")
    print("-" * 35)
    resultats = []
    for rang, (i, score) in enumerate(scores_tries, 1):
        asin_reco = produits['asin'].iloc[i]
        print(f"{rang:<5} {asin_reco:<15} {score:.4f}")
        resultats.append((asin_reco, round(score, 4)))

    return resultats

# Test avec le premier produit de notre liste
asin_test = produits['asin'].iloc[0]
print(f"Produit de départ : {asin_test}\n")
recommander_contenu(asin_test, n_reco=5)

Produit de départ : B005AL5H9S

Produits similaires à B005AL5H9S :

Rang  ASIN            Similarité
-----------------------------------
1     B08BY91SGT      0.4961
2     B08JPK6MKD      0.3584
3     B09NFQ69KT      0.3502
4     B089189NC2      0.3261
5     B07C7S9WNP      0.3014


[('B08BY91SGT', np.float64(0.4961)),
 ('B08JPK6MKD', np.float64(0.3584)),
 ('B09NFQ69KT', np.float64(0.3502)),
 ('B089189NC2', np.float64(0.3261)),
 ('B07C7S9WNP', np.float64(0.3014))]

## Étape 5 — Évaluation (Précision@5)

**Logique :**
1. Pour chaque utilisateur du test, récupérer ses produits aimés dans le train (rating ≥ 4)
2. Recommander des produits similaires à chacun de ces produits aimés
3. Vérifier si le produit réel du test est dans les recommandations

In [ ]:
df_train = pd.read_csv("../data/processed/train.csv")
df_test  = pd.read_csv("../data/processed/test.csv")

hits  = 0
total = 0
asins_disponibles = set(produits['asin'].values)

def recommander_contenu_silent(asin, n_reco=5):
    """Version silencieuse pour l'évaluation (sans print)."""
    if asin not in asins_disponibles:
        return []
    idx = produits[produits['asin'] == asin].index[0]
    scores = list(enumerate(sim[idx]))
    scores_tries = sorted(scores, key=lambda x: x[1], reverse=True)[1:n_reco+1]
    return [produits['asin'].iloc[i] for i, _ in scores_tries]

for _, row_test in df_test.iterrows():
    user_id   = row_test['user_id']
    asin_reel = row_test['asin']

    if asin_reel not in asins_disponibles:
        continue

    aimes = df_train[
        (df_train['user_id'] == user_id) & (df_train['rating'] >= 4)
    ]['asin'].tolist()
    aimes = [a for a in aimes if a in asins_disponibles]

    if not aimes:
        continue

    recos = set()
    for asin_aime in aimes:
        recos.update(recommander_contenu_silent(asin_aime, n_reco=5))

    total += 1
    if asin_reel in recos:
        hits += 1

precision = hits / total * 100 if total > 0 else 0
print(f"Utilisateurs évalués : {total}")
print(f"Hits (produit réel recommandé) : {hits}")
print(f"Précision@5 TF-IDF : {precision:.2f}%")

## Étape 6 — Sauvegarde du modèle

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

# Sauvegarde du vectoriseur TF-IDF
joblib.dump(tfidf, "../models/tfidf_vectorizer.joblib")

# Sauvegarde de la matrice de similarité
np.save("../models/sim_matrix.npy", sim)

# Sauvegarde de la liste des produits (index ASIN)
produits[['asin']].to_csv("../models/produits_index.csv", index=False)

print("Modèles sauvegardés :")
print("  → models/tfidf_vectorizer.joblib")
print("  → models/sim_matrix.npy")
print("  → models/produits_index.csv")

# Vérification du chargement
tfidf_charge   = joblib.load("../models/tfidf_vectorizer.joblib")
sim_charge     = np.load("../models/sim_matrix.npy")
produits_index = pd.read_csv("../models/produits_index.csv")

print(f"\nVérification :")
print(f"  TF-IDF : {type(tfidf_charge).__name__}, {len(tfidf_charge.vocabulary_):,} mots")
print(f"  Matrice similarité : {sim_charge.shape}")
print(f"  Produits index : {len(produits_index):,} produits")

## Conclusion

### Ce qu'on a construit
Un système de recommandation **par filtrage de contenu** basé sur le texte des avis produits.

### Pipeline complet
| Étape | Action | Résultat |
|-------|--------|----------|
| 1 | Chargement + agrégation | 730 produits, 1 texte par produit |
| 2 | Vectorisation TF-IDF | Matrice 730 × 5 000 mots |
| 3 | Similarité cosinus | Matrice 730 × 730 scores |
| 4 | Fonction de recommandation | Top N produits similaires |
| 5 | Évaluation Précision@5 | **12.41%** sur 790 utilisateurs |

### Comparaison des modèles
| Modèle | Précision@5 | Type | Force |
|--------|-------------|------|-------|
| LSA + KNN | 1.24% | Collaboratif | Données denses |
| **TF-IDF** | **12.41%** | **Contenu** | **Cold start, texte riche** |

### Pourquoi TF-IDF est plus performant ici
Le dataset est très sparse (99.18%) avec peu d'avis par utilisateur. Le filtrage par contenu tire parti de la richesse du texte des avis pour trouver des produits similaires, sans dépendre de l'historique d'autres utilisateurs.

### Fichiers sauvegardés
- `models/tfidf_vectorizer.joblib` — le vectoriseur entraîné
- `models/sim_matrix.npy` — la matrice de similarité précalculée
- `models/produits_index.csv` — l'index ASIN des produits